# Неопределённость прогноза и сценарии мощности

Точечный прогноз не показывает диапазон риска. В этом notebook строится state-space модель SARIMAX, рассчитываются 80% и 95% интервалы, а прогноз переводится в сценарии потребности в операторах.

Результаты:

- `outputs/forecast_scenarios.csv`;
- `outputs/charts/forecast_intervals.png`;
- `outputs/charts/capacity_scenarios.png`;
- оценка дефицита мощности для базового, релизного и аварийного сценариев.

> **Место в производственном маршруте:** 4 из 9  
> **Ориентир очного занятия:** 165–180 минут  
> **Режим:** Управляемая практика  
> **Выход этапа:** Интервалы и сценарии мощности

Студенческая версия содержит задания и контрольные точки без полного решения.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw" / "tickets.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Не найдена папка проекта. Убедитесь, что notebook находится внутри распакованного комплекта."
    )


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHART_DIR = OUTPUT_DIR / "charts"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [OUTPUT_DIR, CHART_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Корень проекта:", PROJECT_ROOT)

In [ ]:
source = OUTPUT_DIR / "daily_demand.csv"
if not source.exists():
    raise FileNotFoundError("Сначала выполните notebook 01")

daily = pd.read_csv(source, parse_dates=["created_date"]).sort_values("created_date")
series = daily.set_index("created_date")["tickets_count"].astype(float).asfreq("D")
print(series.shape, series.isna().sum())

## Задание 1. Постройте state-space прогноз

Используйте `SARIMAX` или state-space `ExponentialSmoothing`. Получите объект прогноза через `get_forecast(steps=14)`.

In [ ]:
# Подсказка:
# from statsmodels.tsa.statespace.sarimax import SARIMAX
# model = SARIMAX(...)
# result = model.fit(...)
# forecast_result = result.get_forecast(steps=14)

# TODO: обучите модель и получите прогноз.

## Задание 2. Рассчитайте интервалы 80% и 95%

In [ ]:
# Подсказка:
# ci95 = forecast_result.conf_int(alpha=0.05)
# ci80 = forecast_result.conf_int(alpha=0.20)

# TODO: соберите DataFrame date, forecast, lower_80, upper_80, lower_95, upper_95.

## Задание 3. Создайте три сценария

In [ ]:
scenario_factors = {"base": 1.00, "release": 1.15, "outage": 1.35}

# TODO: умножьте прогноз и интервалы на коэффициент сценария.
# TODO: оцените required_agents по верхней границе 80% интервала.

# Для required_agents используйте AHT, длительность смены и target_utilization.
# Формула: ceil(upper_80 * AHT / (shift_hours * 60 * utilization)).

## Задание 4. Постройте два графика

In [ ]:
# 1. Факт + прогноз + 80/95% интервалы.
# 2. Требуемые операторы по трём сценариям.
# Сохраните графики в outputs/charts.

## Вопросы для вывода

1. Почему интервал расширяется с увеличением горизонта?
2. Какой сценарий создаёт максимальный дефицит мощности?
3. Почему нельзя воспринимать сценарные коэффициенты как доказанную причинность?
4. Какой риск возникает, если планировать персонал только по точечному прогнозу?